# 1.4 Prompt Engineering - 提示词优化工厂

为智能知识库Agent构建高质量的Prompt模板库和自动评估系统

## 项目定位

本实验产出 `prompts/` 目录下的Prompt模板, 这些模板将被6.3最终项目直接使用

In [ ]:
import sys, os
sys.path.insert(0, "..")
from config import get_client, verify_config
from agent_project import *
from agent_project.tools import create_default_registry

client = get_client()
print(f"项目模块已加载 | LLM: {client.name} ({client.model})")


In [ ]:
# Prompt模板库 - 智能知识库Agent的各类Prompt
PROMPT_TEMPLATES = {
    "system": """你是智能知识库助手。
规则:
1. 基于提供的文档内容回答问题, 引用来源
2. 如果文档中没有相关信息, 明确告知用户
3. 回答简洁准确, 不超过500字
4. 对于不确定的内容, 标注"[存疑]"
5. 使用中文回答""",

    "rag_qa": """基于以下知识库内容回答问题:
{context}
问题: {question}
请引用来源编号。如果知识库中没有相关信息, 请明确说明。""",

    "agent_react": """你是智能Agent。在Thought-Action-Observation循环中推理。
可用工具: {tools}
格式:
Thought: 思考过程
Action: 工具名
Action Input: JSON参数
Observation: 工具返回
...
Final Answer: 最终回答""",

    "summarize": """请用2-3句话总结以下对话的核心内容:
{conversation}
摘要:""",
}

print(f"已加载 {len(PROMPT_TEMPLATES)} 个Prompt模板:")
for name in PROMPT_TEMPLATES:
    print(f"  - {name}")


In [ ]:
# Prompt自动评估系统
import json, time

class PromptEvaluator:
    """Prompt质量自动评估器 - A/B测试和评分"""

    def __init__(self, llm_client):
        self.llm = llm_client
        self.results = []

    def evaluate(self, prompt_name, template, test_cases, **kwargs):
        """用测试用例评估Prompt模板"""
        for case in test_cases:
            filled = template.format(**{**case, **kwargs})
            t0 = time.time()
            r = self.llm.chat(filled, temperature=0.3, max_tokens=300)
            elapsed = (time.time() - t0) * 1000
            score = self._score(r["content"], case.get("expected", ""))
            self.results.append({"prompt": prompt_name, "case": case.get("name",""),
                "latency_ms": elapsed, "score": score, "tokens": r["usage"]["output"]})
            print(f"  [{prompt_name}] {case.get('name','?')}: 评分={score}/100, {elapsed:.0f}ms, {r['usage']['output']}t")

    def _score(self, response, expected):
        """简单评分: 长度合理 + 关键词匹配"""
        s = 0
        if 50 < len(response) < 800: s += 40
        elif len(response) > 0: s += 20
        if expected:
            hits = sum(1 for w in expected.split() if w in response)
            s += min(40, hits * 10)
        if any(w in response for w in ["根据", "来源", "知识库", "文档"]): s += 20
        return min(100, s)

    def report(self):
        """生成评估报告"""
        if not self.results: return "无评估数据"
        avg_score = sum(r["score"] for r in self.results) / len(self.results)
        avg_latency = sum(r["latency_ms"] for r in self.results) / len(self.results)
        return f"评估 {len(self.results)} 个案例 | 均分: {avg_score:.0f} | 均延迟: {avg_latency:.0f}ms"

evaluator = PromptEvaluator(client)
print("PromptEvaluator已就绪")


In [ ]:
# 实际测试: 对比不同Prompt的效果
TEST_CASES = [
    {"name": "知识问答", "question": "什么是MCP协议?",
     "context": "MCP(Model Context Protocol)是AI Agent标准化协议, 已捐赠Linux Foundation。",
     "expected": "MCP Agent 协议 标准"},
    {"name": "无相关内容", "question": "今天天气怎么样?",
     "context": "(无相关文档)",
     "expected": "没有 无法 未找到"},
]

print("===== Prompt A/B对比测试 =====\n")
for tc in TEST_CASES:
    print(f"测试: {tc['name']} | 问题: {tc['question']}")
    r = client.chat(
        system=PROMPT_TEMPLATES["system"],
        messages=[{"role": "user", "content": PROMPT_TEMPLATES["rag_qa"].format(**tc)}],
        temperature=0.3, max_tokens=300
    )
    print(f"  回答: {r['content'][:150]}...")
    print()

print("===== 结构化输出测试 =====\n")
r = client.chat(
    system="你是JSON输出专家。只输出纯JSON。",
    messages=[{"role": "user", "content": "分析情感: 这个产品很好用但有点贵。输出JSON: {sentiment, confidence, keywords}"}],
    temperature=0.1, max_tokens=150
)
print(f"JSON输出:\n{r['content']}")


In [ ]:
# 导出Prompt模板供项目使用
import json
os.makedirs("prompts", exist_ok=True)
with open("prompts/templates.json", "w", encoding="utf-8") as f:
    json.dump(PROMPT_TEMPLATES, f, ensure_ascii=False, indent=2)
print("Prompt模板已导出到 prompts/templates.json")
